# Converting the dataset into sequences that can be used by SAKT

In [9]:
## Import the dataset
import pandas as pd
import pickle

from google.colab import drive
drive.mount('/content/drive')

path = "/content/drive/MyDrive/education-ml-research/ASSISTments2009/assistments_sakt_ready.csv"

df = pd.read_csv(path, encoding="latin1")

print(df.head())

print(df.shape)

print(df.columns)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   user_id  problem_id  skill_encoded  correct  order_id
0    64525       51424              0        1  33022537
1    64525       51435              0        1  33022709
2    70363       51444              0        0  35450204
3    70363       51395              0        1  35450295
4    70363       51481              0        0  35450311
(453537, 5)
Index(['user_id', 'problem_id', 'skill_encoded', 'correct', 'order_id'], dtype='object')


## Sort and build sequences

In [10]:
## Sort data chronologically
df = df.sort_values(
    ["user_id", "order_id"]
)

student_sequences = {}

for student, group in df.groupby("user_id"):

    sequence = []

    for _, row in group.iterrows():

        sequence.append({

            "problem_id": int(row["problem_id"]),

            "skill_id": int(row["skill_encoded"]),

            "correct": int(row["correct"])

        })

    student_sequences[student] = sequence


## Temporal Split

In [11]:
train_sequences = {}

validation_sequences = {}

test_sequences = {}

## Split each student's history

for student, sequence in student_sequences.items():

    n = len(sequence)

    train_end = int(0.8 * n)

    val_end = int(0.9 * n)

    train_sequences[student] = sequence[:train_end]

    validation_sequences[student] = sequence[train_end:val_end]

    test_sequences[student] = sequence[val_end:]

## Check the split

In [13]:
student = list(student_sequences.keys())[0]

print("Original:", len(student_sequences[student]))

## 80%
print("Train:", len(train_sequences[student]))

## 10%
print("Validation:", len(validation_sequences[student]))

## 10%
print("Test:", len(test_sequences[student]))

Original: 45
Train: 36
Validation: 4
Test: 5


## Save data

In [22]:
output_path = "/content/drive/MyDrive/education-ml-research/ASSISTments2009/"

## Save training sequences
with open(output_path + "train_sequences.pkl", "wb") as f:
    pickle.dump(train_sequences, f)

## Save validation sequences
with open(output_path + "validation_sequences.pkl", "wb") as f:
    pickle.dump(validation_sequences, f)

## Save test sequences
with open(output_path + "test_sequences.pkl", "wb") as f:
    pickle.dump(test_sequences, f)



## Check that data saved correctly

In [21]:
with open(output_path + "train_sequences.pkl", "rb") as f:
    loaded_train = pickle.load(f)
print(len(loaded_train))
student = list(loaded_train.keys())[0]

## Data should match
print(loaded_train[student][:3])
print(train_sequences[student][:3])

3113
[{'problem_id': 93383, 'skill_id': 1, 'correct': 0}, {'problem_id': 93383, 'skill_id': 24, 'correct': 0}, {'problem_id': 93383, 'skill_id': 43, 'correct': 0}]
[{'problem_id': 93383, 'skill_id': 1, 'correct': 0}, {'problem_id': 93383, 'skill_id': 24, 'correct': 0}, {'problem_id': 93383, 'skill_id': 43, 'correct': 0}]
